# 07. SageMaker Pipeline으로 GR00T Fine-tuning 실행

이 노트북은 GR00T fine-tuning을 SageMaker Pipeline(단일 학습 스텝)으로 실행합니다. 학습 잡이 끝에 압축 해제된 모델을 `s3://<bucket>/<model.s3_prefix>/<execution-id>/`로 직접 업로드하고, 이를 FSx for Lustre로 마운트해 IsaacSim이 로드합니다.

## 선행 조건

- `setup-notebooks.sh` 실행 완료
- 커널 = GR00T (uv)
- `infra/groot` 배포 완료

In [ ]:
from pathlib import Path
import yaml
DOMAIN_ROOT = Path.cwd().parent          # notebooks/ 의 부모 = groot/
CONFIG = yaml.safe_load((DOMAIN_ROOT / "config.yaml").read_text())
aws = CONFIG["aws"]; ecr = CONFIG.get("ecr", {}); model = CONFIG.get("model", {})
train = CONFIG.get("training", {}); mlflow = CONFIG.get("mlflow", {})
BUCKET, REGION, ROLE = aws["bucket_name"], aws.get("region","us-east-1"), aws["role_arn"]
TRAINING_IMAGE_URI = ecr["training_uri"]
assert BUCKET and ROLE and TRAINING_IMAGE_URI, "config.yaml 값 누락 — update-config 먼저 실행"
print(BUCKET, REGION); print(TRAINING_IMAGE_URI)

(선택) 사전 준비: 컨테이너 이미지 빌드 / 데이터셋 업로드. 이미 했다면 건너뜁니다.

In [ ]:
# 학습 컨테이너를 CodeBuild로 빌드 (약 20~40분). 이미 빌드했으면 건너뛰세요.
!cd {DOMAIN_ROOT} && uv run python training/scripts/trigger_build.py --type training

In [ ]:
# leisaac-pick-orange 데이터셋을 S3에 업로드 (v3면 자동 v2.1 변환).
!cd {DOMAIN_ROOT} && uv run python training/data/upload_dataset.py --hf-dataset-id LightwheelAI/leisaac-pick-orange
DATASET_S3_URI = f"s3://{BUCKET}/datasets/leisaac-pick-orange"
print(DATASET_S3_URI)

## 1단계: SageMaker 세션과 파이프라인 파라미터

실행 시 오버라이드할 수 있는 입력들입니다.

In [ ]:
import boto3
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterInteger, ParameterString
session = PipelineSession(boto_session=boto3.Session(region_name=REGION))
p_embodiment_tag = ParameterString(name="EmbodimentTag", default_value="NEW_EMBODIMENT")
p_dataset_s3_uri = ParameterString(name="DatasetS3Uri", default_value=DATASET_S3_URI)
p_instance_type  = ParameterString(name="InstanceType", default_value=train.get("instance_type","ml.g6e.12xlarge"))
p_max_steps      = ParameterInteger(name="MaxSteps", default_value=int(train.get("max_steps",100)))
p_global_batch   = ParameterInteger(name="GlobalBatchSize", default_value=int(train.get("global_batch_size",32)))
p_num_gpus       = ParameterInteger(name="NumGpus", default_value=int(train.get("num_gpus",0) or 0))

## 2단계: 학습 스텝(Estimator + TrainingStep)

checkpoint 경로에 실행 ID를 넣어 충돌을 막고, HF Trainer stdout을 CloudWatch metric으로 파싱합니다. MLflow tracking이 설정돼 있으면 자동 로깅됩니다.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.functions import Join
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.steps import TrainingStep
metric_definitions = [
    {"Name":"train:loss","Regex":r"'loss':\s*([0-9.eE+-]+)"},
    {"Name":"eval:loss","Regex":r"'eval_loss':\s*([0-9.eE+-]+)"},
]
checkpoint_s3 = Join(on="/", values=[f"s3://{BUCKET}/checkpoints", ExecutionVariables.PIPELINE_EXECUTION_ID])
model_prefix = (model.get("s3_prefix","models/groot-sm") or "models/groot-sm").strip("/")
export_s3_uri = Join(on="/", values=[f"s3://{BUCKET}/{model_prefix}", ExecutionVariables.PIPELINE_EXECUTION_ID])
env = {"SM_HP_WANDB_API_KEY": "ssm:/groot/wandb-key"}
if mlflow.get("tracking_server_arn"):
    env.update({"MLFLOW_TRACKING_URI": mlflow["tracking_server_arn"],
                "MLFLOW_EXPERIMENT_NAME": mlflow.get("experiment_name","groot-sm-finetune"),
                "HF_MLFLOW_LOG_ARTIFACTS": "true"})
estimator = Estimator(
    image_uri=TRAINING_IMAGE_URI, role=ROLE, entry_point="train.py",
    source_dir=str(DOMAIN_ROOT / "training" / "container"),
    instance_type=p_instance_type, instance_count=1,
    output_path=f"s3://{BUCKET}/output", checkpoint_s3_uri=checkpoint_s3,
    metric_definitions=metric_definitions,
    hyperparameters={"embodiment_tag": p_embodiment_tag, "max_steps": p_max_steps,
                     "global_batch_size": p_global_batch,
                     "save_steps": str(train.get("save_steps",50)), "num_gpus": p_num_gpus,
                     "export_s3_uri": export_s3_uri},
    sagemaker_session=session, environment=env)
training_step = TrainingStep(name="GR00TFinetune", estimator=estimator,
    inputs={"dataset": TrainingInput(s3_data=p_dataset_s3_uri)})

## 3단계: 모델 export (source에서 직접 업로드)

학습 잡이 끝에 압축 해제된 체크포인트를 `s3://<bucket>/<model.s3_prefix>/<execution-id>/`로 직접 업로드합니다(`export_s3_uri` hyperparameter). 별도의 처리 단계 없이 source에서 바로 내보내므로 재다운로드/재압축해제가 없습니다.

## 4단계: 파이프라인 조립 + 업서트(정의 등록)

`upsert`는 실행하지 않고 정의만 만듭니다.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
alias = aws.get("alias","") or ""
pipeline = Pipeline(name=f"groot-sm-finetuning{('-'+alias) if alias else ''}",
    parameters=[p_embodiment_tag,p_dataset_s3_uri,p_instance_type,p_max_steps,p_global_batch,p_num_gpus],
    steps=[training_step], sagemaker_session=session)
pipeline.upsert(role_arn=ROLE)
print("업서트 완료:", pipeline.name)

## 5단계: 실행

파라미터를 명시 전달해 시작합니다.

In [ ]:
execution = pipeline.start(parameters={
    "EmbodimentTag":"NEW_EMBODIMENT", "DatasetS3Uri": DATASET_S3_URI,
    "MaxSteps": int(train.get("max_steps",100)), "GlobalBatchSize": int(train.get("global_batch_size",32))})
print("실행 ARN:", execution.arn)

In [ ]:
execution.describe()["PipelineExecutionStatus"]

## 완료 후 안내

- 학습 잡이 끝에 압축 해제된 모델을 별도 스텝 없이 `s3://<bucket>/<model_prefix>/<execution-id>/`로 직접 업로드합니다.
- SageMaker 콘솔 Pipelines 탭에서 실행 상태를 확인할 수 있습니다.
- MLflow tracking server 링크에서 학습 메트릭을 확인할 수 있습니다.